# 📊 Cuaderno de Trabajo 1: Fundamentos de Business Intelligence (BI)
## De los Datos en Bruto al Data Warehouse ( Data Lake) y KPIs Estratégicos

**Asignatura:** Inteligencia de Negocios  
**Nivel:** Pregrado / Posgrado  
**Objetivo Práctico:** Implementar un flujo completo de BI en Python: Simulación de Data Lake ➔ Proceso ETL ➔ Modelado multidimensional en estrella en Data Warehouse local (SQL) ➔ Cálculo de KPIs de proyectos ➔ Dashboard interactiva con Plotly.

---

### 🎯 Contenido del Cuaderno:
1. **Capa 1: Data Lake (Fuentes Crudas Heterogéneas)**: Generación e inspección de logs JSON, datos de proyectos en CSV y datos de recursos.
2. **Capa 2: ETL (Extracción, Limpieza y Transformación)**: Normalización, manejo de valores nulos y estructuración en Modelo en Estrella (*Star Schema*).
3. **Capa 3: Data Warehouse (Almacenamiento SQL)**: Creación de base de datos relacional analítica (`sqlite3` / `duckdb`) y consultas SQL de negocio.
4. **Capa 4: Analítica de Negocio & KPIs**: Cálculo de ROI, Desviación de Presupuesto, Margen Operativo e Indicadores de Rendimiento.
5. **Capa 5: Visualización Executive Dashboard**: Gráficos interactivos en Plotly.
6. **Laboratorio & Desafíos Prácticos**: Ejercicios propuestos para el estudiante.

## ⚙️ 0. Configuración e Importación de Librerías
Asegúrate de contar con las librerías principales de análisis y visualización.

In [1]:
import os
import json
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Configurar visualización de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("[OK] Librerías cargadas exitosamente.")

[OK] Librerías cargadas exitosamente.


---
## 🗄️ 1. Capa Data Lake: Generación e Inspección de Fuentes Crudas

Simularemos un **Data Lake** donde los datos llegan en formatos crudos y no estructurados desde distintas áreas de la empresa:
- `raw_projects.csv`: Datos de proyectos, presupuestos y clientes.
- `raw_resources.csv`: Información sobre el personal y su costo por hora.
- `raw_time_logs.json`: Logs semiestructurados de horas trabajadas registradas en el sistema.

In [2]:
# Crear carpeta del Data Lake simulado
os.makedirs('data_lake', exist_ok=True)

# 1. Generar raw_projects.csv
df_projects_raw = pd.DataFrame({
    'project_id': ['PRJ-101', 'PRJ-102', 'PRJ-103', 'PRJ-104', 'PRJ-105', 'PRJ-106'],
    'project_name': ['Migración ERP Cloud', 'App Móvil V2', 'Rediseño Plataforma B2B', 'Sistema BI & Analytics', 'Portal Clientes', 'Auditoría Seguridad'],
    'client_name': ['Fintech Global', 'Retail Co', 'Logística Express', 'Fintech Global', 'Salud Integral', 'Retail Co'],
    'budget_usd': [150000.00, 85000.00, 120000.00, 95000.00, 60000.00, 45000.00],
    'start_date': ['2026-01-15', '2026-02-01', '2026-01-10', '2026-03-01', '2026-02-15', '2026-04-01'],
    'planned_hours': [1200, 750, 1000, 800, 500, 350]
})
df_projects_raw.to_csv('data_lake/raw_projects.csv', index=False)

# 2. Generar raw_resources.csv
df_resources_raw = pd.DataFrame({
    'resource_id': ['RES-01', 'RES-02', 'RES-03', 'RES-04', 'RES-05'],
    'resource_name': ['Ana Martínez', 'Carlos López', 'Beatriz Gómez', 'David Ruiz', 'Elena Torres'],
    'role': ['Data Architect', 'Senior Developer', 'Project Manager', 'BI Analyst', 'DevOps Specialist'],
    'hourly_rate_usd': [95.00, 80.00, 85.00, 70.00, 90.00]
})
df_resources_raw.to_csv('data_lake/raw_resources.csv', index=False)

# 3. Generar raw_time_logs.json (Log crudo semiestructurado)
time_logs_raw = [
    {"log_id": 1, "project_id": "PRJ-101", "resource_id": "RES-01", "hours_logged": 160, "date": "2026-02-28", "status": "Approved"},
    {"log_id": 2, "project_id": "PRJ-101", "resource_id": "RES-02", "hours_logged": 210, "date": "2026-02-28", "status": "Approved"},
    {"log_id": 3, "project_id": "PRJ-101", "resource_id": "RES-03", "hours_logged": 180, "date": "2026-02-28", "status": "Approved"},
    {"log_id": 4, "project_id": "PRJ-102", "resource_id": "RES-02", "hours_logged": 350, "date": "2026-03-15", "status": "Approved"},
    {"log_id": 5, "project_id": "PRJ-102", "resource_id": "RES-03", "hours_logged": 200, "date": "2026-03-15", "status": "Approved"},
    {"log_id": 6, "project_id": "PRJ-103", "resource_id": "RES-01", "hours_logged": 450, "date": "2026-03-31", "status": "Approved"},
    {"log_id": 7, "project_id": "PRJ-103", "resource_id": "RES-04", "hours_logged": 580, "date": "2026-03-31", "status": "Approved"},
    {"log_id": 8, "project_id": "PRJ-104", "resource_id": "RES-04", "hours_logged": 300, "date": "2026-04-15", "status": "Approved"},
    {"log_id": 9, "project_id": "PRJ-104", "resource_id": "RES-05", "hours_logged": 320, "date": "2026-04-15", "status": "Approved"},
    {"log_id": 10, "project_id": "PRJ-105", "resource_id": "RES-02", "hours_logged": 220, "date": "2026-04-30", "status": "Approved"},
    {"log_id": 11, "project_id": "PRJ-106", "resource_id": "RES-05", "hours_logged": 180, "date": "2026-04-30", "status": "Approved"}
]

with open('data_lake/raw_time_logs.json', 'w') as f:
    json.dump(time_logs_raw, f, indent=2)

print("[OK] Fuentes del Data Lake creadas exitosamente en ./data_lake/")

[OK] Fuentes del Data Lake creadas exitosamente en ./data_lake/


### Visualizar los Datos Crudos del Data Lake

In [3]:
# Leer y visualizar contenido del Data Lake
df_proj = pd.read_csv('data_lake/raw_projects.csv')
df_res = pd.read_csv('data_lake/raw_resources.csv')
with open('data_lake/raw_time_logs.json') as f:
    json_logs = json.load(f)
df_logs = pd.DataFrame(json_logs)

print("--- PROYECTOS (CSV) ---")
print(df_proj.head())

print("\n--- RECURSOS (CSV) ---")
print(df_res.head())

print("\n--- LOGS DE HORAS (JSON) ---")
print(df_logs.head())

--- PROYECTOS (CSV) ---
  project_id             project_name        client_name  budget_usd  \
0    PRJ-101      Migración ERP Cloud     Fintech Global   150000.00   
1    PRJ-102             App Móvil V2          Retail Co    85000.00   
2    PRJ-103  Rediseño Plataforma B2B  Logística Express   120000.00   
3    PRJ-104   Sistema BI & Analytics     Fintech Global    95000.00   
4    PRJ-105          Portal Clientes     Salud Integral    60000.00   

   start_date  planned_hours  
0  2026-01-15           1200  
1  2026-02-01            750  
2  2026-01-10           1000  
3  2026-03-01            800  
4  2026-02-15            500  

--- RECURSOS (CSV) ---
  resource_id  resource_name               role  hourly_rate_usd
0      RES-01   Ana Martínez     Data Architect            95.00
1      RES-02   Carlos López   Senior Developer            80.00
2      RES-03  Beatriz Gómez    Project Manager            85.00
3      RES-04     David Ruiz         BI Analyst            70.00
4      R

---
## 🔄 2. Capa ETL: Transformación y Modelado Multidimensional (Star Schema)

En la etapa ETL:
1. **Extraemos** los datos de los 3 archivos.
2. **Transformamos** calculando el costo real de horas invertidas (`hours_logged * hourly_rate_usd`).
3. **Construimos el Esquema en Estrella (*Star Schema*)** para el Data Warehouse:
   - **Tabla de Hechos (`fact_project_execution`)**: Almacena las métricas numéricas acumuladas por proyecto.
   - **Tablas de Dimensión**: `dim_project`, `dim_resource`, `dim_client`.

In [4]:
# 1. Unir logs con recursos para calcular costo real por registro
df_merged_logs = df_logs.merge(df_res, on='resource_id', how='left')
df_merged_logs['actual_cost_usd'] = df_merged_logs['hours_logged'] * df_merged_logs['hourly_rate_usd']

# 2. Agrupar costos y horas reales por proyecto
df_project_actuals = df_merged_logs.groupby('project_id').agg(
    total_hours_logged=('hours_logged', 'sum'),
    total_actual_cost_usd=('actual_cost_usd', 'sum')
).reset_index()

# 3. Unir con la tabla de proyectos para construir la Tabla de Hechos
fact_project_execution = df_proj.merge(df_project_actuals, on='project_id', how='left')
fact_project_execution['total_hours_logged'] = fact_project_execution['total_hours_logged'].fillna(0)
fact_project_execution['total_actual_cost_usd'] = fact_project_execution['total_actual_cost_usd'].fillna(0)

# 4. Métricas de variación
fact_project_execution['cost_variance_usd'] = fact_project_execution['total_actual_cost_usd'] - fact_project_execution['budget_usd']
fact_project_execution['hour_variance'] = fact_project_execution['total_hours_logged'] - fact_project_execution['planned_hours']
fact_project_execution['budget_overrun_pct'] = (fact_project_execution['cost_variance_usd'] / fact_project_execution['budget_usd']) * 100

print("[OK] Tabla de Hechos construida exitosamente.")
print(fact_project_execution)

[OK] Tabla de Hechos construida exitosamente.
  project_id             project_name        client_name  budget_usd  \
0    PRJ-101      Migración ERP Cloud     Fintech Global   150000.00   
1    PRJ-102             App Móvil V2          Retail Co    85000.00   
2    PRJ-103  Rediseño Plataforma B2B  Logística Express   120000.00   
3    PRJ-104   Sistema BI & Analytics     Fintech Global    95000.00   
4    PRJ-105          Portal Clientes     Salud Integral    60000.00   
5    PRJ-106      Auditoría Seguridad          Retail Co    45000.00   

   start_date  planned_hours  total_hours_logged  total_actual_cost_usd  \
0  2026-01-15           1200                 550               47300.00   
1  2026-02-01            750                 550               45000.00   
2  2026-01-10           1000                1030               83350.00   
3  2026-03-01            800                 620               49800.00   
4  2026-02-15            500                 220               17600.00   

---
## 🏛️ 3. Capa Data Warehouse: Carga y Consultas SQL (`SQLite` / `DuckDB`)

Cargaremos las tablas modeladas en nuestro **Data Warehouse local en SQL** (`bi_warehouse.db`). Esto refleja cómo un analista de BI consulta las tablas mediante SQL relacional.

In [7]:
# Conectar/Crear base de datos SQLite como Data Warehouse
conn = sqlite3.connect('bi_warehouse.db')

# Cargar tablas relacionales
fact_project_execution.to_sql('fact_project_execution', conn, if_exists='replace', index=False)
df_res.to_sql('dim_resource', conn, if_exists='replace', index=False)
df_proj[['project_id', 'project_name', 'client_name', 'start_date']].to_sql('dim_project', conn, if_exists='replace', index=False)

print("[OK] Data Warehouse cargado en bi_warehouse.db")

[OK] Data Warehouse cargado en bi_warehouse.db


### Consulta SQL 1: Proyectos con Mayor Sobrecosto Presupuestario

In [8]:
query_overrun = """
SELECT 
    project_name,
    client_name,
    budget_usd,
    total_actual_cost_usd,
    cost_variance_usd,
    ROUND(budget_overrun_pct, 2) AS overrun_pct
FROM fact_project_execution
WHERE cost_variance_usd > 0
ORDER BY cost_variance_usd DESC;
"""

df_sql_overrun = pd.read_sql_query(query_overrun, conn)
print(df_sql_overrun)

Empty DataFrame
Columns: [project_name, client_name, budget_usd, total_actual_cost_usd, cost_variance_usd, overrun_pct]
Index: []


### Consulta SQL 2: Análisis por Cliente (Ingresos vs Costo Total)

In [9]:
query_client = """
SELECT 
    client_name,
    COUNT(project_id) AS total_projects,
    SUM(budget_usd) AS total_budget_usd,
    SUM(total_actual_cost_usd) AS total_actual_cost_usd,
    SUM(budget_usd - total_actual_cost_usd) AS net_margin_usd
FROM fact_project_execution
GROUP BY client_name
ORDER BY total_budget_usd DESC;
"""

df_sql_client = pd.read_sql_query(query_client, conn)
print(df_sql_client)

         client_name  total_projects  total_budget_usd  total_actual_cost_usd  \
0     Fintech Global               2         245000.00               97100.00   
1          Retail Co               2         130000.00               61200.00   
2  Logística Express               1         120000.00               83350.00   
3     Salud Integral               1          60000.00               17600.00   

   net_margin_usd  
0       147900.00  
1        68800.00  
2        36650.00  
3        42400.00  


---
## 📈 4. Cálculo de KPIs Estratégicos de BI

Calculemos los Indicadores Clave de Rendimiento (**KPIs**) del negocio a nivel global:

In [10]:
# KPIs Generales de la Cartera de Proyectos
total_budget = fact_project_execution['budget_usd'].sum()
total_cost = fact_project_execution['total_actual_cost_usd'].sum()
total_variance = total_cost - total_budget
overall_roi = ((total_budget - total_cost) / total_cost) * 100 if total_cost > 0 else 0
projects_overbudget = (fact_project_execution['cost_variance_usd'] > 0).sum()

print("=== SUMMARY DE KPIs EJECUTIVOS ===")
print(f"Presupuesto Total Planificado:  ${total_budget:,.2f} USD")
print(f"Costo Real Incurrido:          ${total_cost:,.2f} USD")
print(f"Desviacion Global Neto:        ${total_variance:,.2f} USD")
print(f"Margen / ROI Global Estimado:  {overall_roi:.2f}%")
print(f"Proyectos sobre Presupuesto:   {projects_overbudget} de {len(fact_project_execution)}")

=== SUMMARY DE KPIs EJECUTIVOS ===
Presupuesto Total Planificado:  $555,000.00 USD
Costo Real Incurrido:          $259,250.00 USD
Desviacion Global Neto:        $-295,750.00 USD
Margen / ROI Global Estimado:  114.08%
Proyectos sobre Presupuesto:   0 de 6


---
## 📊 5. Capa de Visualización: Executive Dashboard (Plotly)

Crearemos tableros gráficos interactivos con `plotly` para presentar los hallazgos ante la dirección gerencial.

### Gráfico 1: Presupuesto vs Costo Real por Proyecto

In [11]:
fig1 = go.Figure()

fig1.add_trace(go.Bar(
    x=fact_project_execution['project_name'],
    y=fact_project_execution['budget_usd'],
    name='Presupuesto Planificado',
    marker_color='#00e5ff'
))

fig1.add_trace(go.Bar(
    x=fact_project_execution['project_name'],
    y=fact_project_execution['total_actual_cost_usd'],
    name='Costo Real Incurrido',
    marker_color='#ec4899'
))

fig1.update_layout(
    title='Presupuesto Planificado vs Costo Real por Proyecto',
    barmode='group',
    template='plotly_dark',
    xaxis_title='Proyecto',
    yaxis_title='Monto en USD ($)',
    height=500
)

fig1.show()

### Gráfico 2: Distribución del Presupuesto por Cliente

In [12]:
fig2 = px.pie(
    df_sql_client, 
    names='client_name', 
    values='total_budget_usd',
    title='Distribucion de Cartera de Presupuesto por Cliente',
    hole=0.4,
    template='plotly_dark',
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig2.update_traces(textinfo='percent+label')
fig2.show()

### Gráfico 3: Matriz de Eficiencia (Horas Planificadas vs Horas Reales)

In [13]:
fig3 = px.scatter(
    fact_project_execution,
    x='planned_hours',
    y='total_hours_logged',
    size='budget_usd',
    color='client_name',
    hover_name='project_name',
    title='Desviacion de Horas: Horas Planificadas vs Registradas',
    labels={'planned_hours': 'Horas Planificadas', 'total_hours_logged': 'Horas Reales Logs'},
    template='plotly_dark'
)

# Añadir línea de referencia 1:1
fig3.add_shape(
    type="line", x0=0, y0=0, x1=1300, y1=1300,
    line=dict(color="Gray", width=2, dash="dash")
)

fig3.show()

---
## ✏️ 6. Laboratorio & Desafíos Prácticos para el Estudiante

### Desafío 1:
Calcula la tarifa promedio por hora real pagada en cada proyecto ($Costo Real / Horas Reales$). ¿Qué proyecto tuvo el costo por hora más elevado?

### Desafío 2:
Escribe una consulta SQL sobre `bi_warehouse.db` que retorne únicamente aquellos proyectos cuyo margen neto ($Presupuesto - Costo Real$) sea mayor a $10,000 USD.

### Desafío 3:
Agrega una nueva columna al DataFrame `fact_project_execution` llamada `status_kpi` con la siguiente regla:
- `'Crítico'`: si `budget_overrun_pct > 15%`
- `'Alerta'`: si `0% < budget_overrun_pct <= 15%`
- `'Saludable'`: si `budget_overrun_pct <= 0%`